In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib
import os

# Load Dataset

In [2]:
df = pd.read_csv("../data/dataset_pupuk.csv")
# df = pd.read_csv("../data/sintetis/dataset_pupuk.csv")
print(f"Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Dataset: 1777 baris, 8 kolom


,nitrogen,fosfor,kalium,plant_age,fase,ec,soil_moisture,recommendation
0,91,190,102,64,2,17.50,82.5,10
1,93,187,101,64,2,17.50,80.5,10
2,91,190,101,64,2,17.50,77.9,10
3,94,188,103,64,2,17.50,78.2,10
4,93,189,100,64,2,17.58,75.1,10


# Definisi fitur dan label

In [3]:
FEATURES = ["nitrogen", "fosfor", "kalium", "plant_age", "fase",
            "ec", "soil_moisture"]

LABEL_NAMES = {
    0: "Tidak perlu",
    1: "Urea/ZA",
    2: "SP-36",
    3: "KCl",
    4: "Urea/ZA + SP-36",
    5: "Urea/ZA + KCl",
    6: "SP-36 + KCl",
    7: "Urea/ZA + SP-36 + KCl",
    8: "NPK 15-15-15",
    9: "Kurangi pemupukan N",
    10: "Flush air (EC/nutrisi tinggi)",
}

X = df[FEATURES].values
y = df["recommendation"].values

# Cek Distribusi Label (KRITIS)

In [4]:
print(f"Jumlah kelas unik: {len(set(y))}")
print(f"\nDistribusi label:")
dist = pd.Series(y).value_counts().sort_index()
for val, count in dist.items():
    print(f"  {val} = {LABEL_NAMES.get(val, '?'):35s} {count:5d} ({count/len(y)*100:.1f}%)")

if len(set(y)) < 2:
    print("\n⚠️  BAHAYA: Cuma 1 kelas! Model gak bisa dilatih.")
    print("    Data kamu perlu variasi kondisi NPK/fase lebih banyak.")

Jumlah kelas unik: 2

Distribusi label:
  7 = Urea/ZA + SP-36 + KCl                   9 (0.5%)
  10 = Flush air (EC/nutrisi tinggi)        1768 (99.5%)


# Split Train/Test

In [5]:
# stratify butuh tiap kelas minimal 2 sampel; kalau error, hapus stratify
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
except ValueError:
    print("Stratify gagal (ada kelas <2 sampel), pakai split biasa.")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

print(f"Train: {len(X_train)} baris")
print(f"Test:  {len(X_test)} baris")

Train: 1421 baris
Test:  356 baris


# Training Random Forest

In [6]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)

model.fit(X_train, y_train)
print("Training selesai.")

Training selesai.


# Cross Validation

In [7]:
n_splits = min(5, pd.Series(y_train).value_counts().min())
if n_splits < 2:
    print("Data per kelas terlalu sedikit untuk cross-validation.")
else:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_weighted")
    print(f"CV F1 (weighted): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

CV F1 (weighted): 1.0000 (+/- 0.0000)


# Evaluasi

In [8]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print()
labels_present = sorted(set(y_test) | set(y_pred))
target_names = [LABEL_NAMES.get(l, str(l)) for l in labels_present]
print(classification_report(y_test, y_pred, labels=labels_present,
                            target_names=target_names, zero_division=0))

Accuracy: 1.0000
F1 (weighted): 1.0000

                               precision    recall  f1-score   support

        Urea/ZA + SP-36 + KCl       1.00      1.00      1.00         2
Flush air (EC/nutrisi tinggi)       1.00      1.00      1.00       354

                     accuracy                           1.00       356
                    macro avg       1.00      1.00      1.00       356
                 weighted avg       1.00      1.00      1.00       356



# Feature Importance

In [9]:
importances = model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]

print("Feature Importance:")
for idx in sorted_idx:
    bar = "█" * int(importances[idx] * 50)
    print(f"  {FEATURES[idx]:15s} {importances[idx]:.4f}  {bar}")

Feature Importance:
  kalium          0.2080  ██████████
  fosfor          0.2068  ██████████
  ec              0.2051  ██████████
  nitrogen        0.2004  ██████████
  soil_moisture   0.1724  ████████
  plant_age       0.0073  
  fase            0.0000  


# Tes Prediksi

In [10]:
sample = {
    "nitrogen": 40,
    "fosfor": 40,
    "kalium": 100,
    "plant_age": 61,
    "fase": 1,
    "ec": 2.0,
    "soil_moisture": 65,
}
X_sample = np.array([[sample[f] for f in FEATURES]])
pred = int(model.predict(X_sample)[0])
conf = float(model.predict_proba(X_sample)[0].max())

print(f"Prediksi: {pred} = {LABEL_NAMES.get(pred)}")
print(f"Confidence: {conf:.2%}")


Prediksi: 7 = Urea/ZA + SP-36 + KCl
Confidence: 61.67%


# Simpan Model

In [11]:
os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/rf_pupuk.joblib")
print("Model tersimpan: models/rf_pupuk.joblib")

Model tersimpan: models/rf_pupuk.joblib
